# 🧠 EX64: Callback Hooks และการบันทึก Metric แบบกำหนดเอง

YOLO เรียกใช้ callable Python ณ จุดสำคัญในวงจรการเทรน

| Hook | ทำงานเมื่อ |
|------|-----------|
| `on_train_start` | ก่อน epoch แรก |
| `on_train_epoch_end` | หลัง train loss, **ก่อน** val |
| `on_fit_epoch_end` | หลัง **ทั้ง** train + val |
| `on_val_end` | หลัง `model.val()` |
| `on_train_end` | หลังทุก epoch |

**สำคัญ:** ใช้ `on_fit_epoch_end` (ไม่ใช่ `on_train_epoch_end`) เพื่ออ่าน `metrics/mAP50(B)` —
key นั้นจะมีค่า **หลัง** validation เท่านั้น

## 🔗 ลิงก์
- [[EX53_Training_Settings_TH]] | [[YOLO_Learning_Plan]]


In [ ]:
# Back up or checkpoint this section of code before starting to modify the large file.
import gc, torch
import matplotlib.pyplot as plt
from ultralytics import YOLO
from solution import register_custom_metric_callback
%matplotlib inline

device = "0" if torch.cuda.is_available() else "cpu"
print(f"[INFO] ใช้อุปกรณ์: {device}")

print("\n--- เริ่มการตรวจสอบ ---")
model = YOLO("yolo11n.pt")
print(f"ก่อนลงทะเบียน: {model.callbacks.get('on_val_end', [])}")

model = register_custom_metric_callback(model)
post_cbs = model.callbacks.get("on_val_end", [])
print(f"หลังลงทะเบียน: {[cb.__name__ for cb in post_cbs]}")
print(f"Hooks ทั้งหมด: {sum(len(v) for v in model.callbacks.values())}")

print("\nรัน model.val() บน coco8 เพื่อ trigger callback...")
try:
    results = model.val(data="coco8.yaml", imgsz=320, device=device, verbose=False)
    mAP50=results.box.map50; mAP95=results.box.map; mp=results.box.mp; mr=results.box.mr
    print(f"  Precision={mp:.4f}  Recall={mr:.4f}")
    print(f"  mAP@0.5={mAP50:.4f}  mAP@0.5:0.95={mAP95:.4f}")
except Exception as e:
    print(f"Validation error: {e}")
    mAP50=mAP95=mp=mr=0.0
print("--- สิ้นสุดการตรวจสอบ ---")

fig, axes = plt.subplots(1,2,figsize=(10,4))
axes[0].bar(["mAP@0.5","mAP@0.5:0.95"],[mAP50,mAP95],color=["#27ae60","#145a32"],edgecolor="black")
axes[0].set_ylim(0,1.0); axes[0].set_ylabel("คะแนน"); axes[0].set_title("mAP (ผ่าน callback)")
for i,v in enumerate([mAP50,mAP95]): axes[0].text(i, v+0.02, f"{v:.3f}", ha="center", fontweight="bold")
axes[1].bar(["Precision","Recall"],[mp,mr],color=["#2980b9","#8e44ad"],edgecolor="black")
axes[1].set_ylim(0,1.0); axes[1].set_ylabel("คะแนน"); axes[1].set_title("P & R")
for i,v in enumerate([mp,mr]): axes[1].text(i, v+0.02, f"{v:.3f}", ha="center", fontweight="bold")
plt.tight_layout(); plt.show()

del model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
